# SE(3)-Transformer Overview

The SE(3)-Transformer is a Graph Neural Network using a variant of self-attention for 3D points and graphs processing. This model is equivariant under continuous 3D roto-translations, meaning that when the inputs (graphs or sets of points) rotate in 3D space (or more generally experience a proper rigid transformation), the model outputs either stay invariant or transform with the input.

# Training

We trained the model using the following script:

```bash
bash scripts/train.sh
```

This script produces a trained model checkpoint: **`model_qm9.pth`**.
> **Note:** This file is a PyTorch model checkpoint containing the trained parameters of the SE(3)-Transformer.

## `train.sh` Script Overview

The `train.sh` script accepts several optional arguments with default values:

```bash
BATCH_SIZE=${1:-240}       # Number of samples per batch
AMP=${2:-true}             # Automatic Mixed Precision (for faster training)
NUM_EPOCHS=${3:-100}       # Number of training epochs
LEARNING_RATE=${4:-0.002}  # Learning rate for optimizer
WEIGHT_DECAY=${5:-0.1}     # Weight decay for regularization
```

### Task Selection

You can specify the target property for prediction using the `TASK` variable. Available options include:

```
'mu', 'alpha', 'homo', 'lumo', 'gap', 'r2', 'zpve', 
'U0', 'U', 'H', 'G', 'Cv', 
'U0_atom', 'U_atom', 'H_atom', 'G_atom', 'A', 'B', 'C'
```

Example:  
```bash
TASK=homo
```

## Python Training Command

The script internally runs the following Python command:

```bash
python -m se3_transformer.runtime.training \
  --amp "$AMP" \
  --batch_size "$BATCH_SIZE" \
  --epochs "$NUM_EPOCHS" \
  --lr "$LEARNING_RATE" \
  --weight_decay "$WEIGHT_DECAY" \
  --use_layer_norm \
  --norm \
  --save_ckpt_path model_qm9.pth \
  --precompute_bases \
  --seed 42 \
  --task "$TASK"
```

**Explanation of Key Flags:**

- `--amp`: Enables mixed-precision training for speed and memory efficiency.  
- `--batch_size`: Number of samples per batch.  
- `--epochs`: Number of training iterations over the entire dataset.  
- `--lr`: Learning rate.  
- `--weight_decay`: Regularization to prevent overfitting.  
- `--use_layer_norm` & `--norm`: Apply layer normalization for stable training.  
- `--save_ckpt_path`: Path to save the trained model checkpoint.  
- `--precompute_bases`: Precomputes geometric bases for faster training.  
- `--seed`: Ensures reproducibility.  
- `--task`: The molecular property to predict.

## Run the Training Script

In this step, we'll run the training script. To make the training process quicker for demonstration purposes, we've set the number of epochs to **10**. 

Running for more epochs may take longer, but will typically yield better accuracy

In [ ]:
bash scripts/train.sh 240 --amp 10

# Inference

We run the model using the following script:

```bash
./scripts/predict.sh
```

This script runs inference using the trained model checkpoint **`model_qm9.pth`** produced during training.

> **Note:** The checkpoint contains all learned model parameters and is required to perform predictions.

## `predict.sh` Script Overview

The `predict.sh` script accepts optional arguments with default values:

```bash
BATCH_SIZE=${1:-240}  # Number of samples per batch
AMP=${2:-true}         # Automatic Mixed Precision
```

### Task Selection

You can specify the target property for prediction using the `TASK` variable. Options include:

```
'mu', 'alpha', 'homo', 'lumo', 'gap', 'r2', 'zpve', 
'U0', 'U', 'H', 'G', 'Cv', 
'U0_atom', 'U_atom', 'H_atom', 'G_atom', 'A', 'B', 'C'
```

Example:
```bash
TASK=homo
```

## Python Inference Command

The script internally runs the following Python command:

```bash
python -m torch.distributed.run --nnodes=1 --nproc_per_node=gpu --max_restarts 0 --module \
  se3_transformer.runtime.inference \
  --amp "$AMP" \
  --batch_size "$BATCH_SIZE" \
  --use_layer_norm \
  --norm \
  --load_ckpt_path model_qm9.pth \
  --task "$TASK"
```

**Explanation of Key Flags:**

- `--amp`: Enables mixed-precision inference.  
- `--batch_size`: Number of samples per batch.  
- `--use_layer_norm` & `--norm`: Apply layer normalization.  
- `--load_ckpt_path`: Path to the trained model checkpoint (`model_qm9.pth`).  
- `--task`: The molecular property to predict.  

> You can modify the `BATCH_SIZE`, `AMP`, and `TASK` variables in the cell to experiment with different settings.

In [ ]:
bash scripts/predict.sh

## Conclusion

In this notebook, we demonstrated the complete SE(3)-Transformer pipeline — from training to prediction. The dataset size was kept the same, but the number of epochs was reduced to build a quick MVP and validate the model’s functionality.

After training, we successfully ran predictions using the trained model, confirming that the pipeline works end-to-end. While this configuration prioritizes speed, extending the training to more epochs and fine-tuning hyperparameters can further improve prediction accuracy and model generalization.

With the workflow now validated, this setup provides a strong foundation for scaling up experiments, benchmarking performance, and adapting the SE(3)-Transformer to more complex or domain-specific datasets.